# Fused residual-add RMSNorm and SwiGLU, plugged into PyTorch

Write two fused kernels every Llama-style model can use (residual add plus RMSNorm, and SwiGLU), count the memory traffic they save, register them as PyTorch custom operators that work with torch.compile and CUDA graphs, test them properly across shapes and data types, and compare them with eager PyTorch and with torch.compile's own generated kernels.

This notebook holds the code from the lesson **[Fused residual-add RMSNorm and SwiGLU, plugged into PyTorch](https://inferenceacademy.com/learn/kern-triton/fused-rmsnorm-and-swiglu-plugged-into-pytorch)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## Kernel 1: residual add plus RMSNorm

Illustration from the lesson (not meant to run on its own):

```python
h = x + residual          # residual add
y = rms_norm(h) * w       # normalise for the next sub-layer
residual = h              # the next add needs h
```

In [ ]:
import torch, triton, triton.language as tl

@triton.jit
def add_rmsnorm_kernel(x_ptr, r_ptr, w_ptr, y_ptr, h_ptr, stride, n_cols, eps, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols
    x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)
    r = tl.load(r_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)
    h = x + r
    tl.store(h_ptr + row * stride + cols, h.to(h_ptr.dtype.element_ty), mask=mask)
    inv = tl.rsqrt(tl.sum(h * h, axis=0) / n_cols + eps)
    w = tl.load(w_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    tl.store(y_ptr + row * stride + cols, (h * inv * w).to(y_ptr.dtype.element_ty), mask=mask)

## Kernel 2: SwiGLU

Illustration from the lesson (not meant to run on its own):

```python
gate, up = gate_up.chunk(2, dim=-1)       # views, no copy
out = torch.nn.functional.silu(gate) * up
```

In [ ]:
@triton.jit
def swiglu_kernel(gu_ptr, out_ptr, n_inter, stride_gu, stride_out, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.program_id(1) * BLOCK + tl.arange(0, BLOCK)
    mask = cols < n_inter
    g = tl.load(gu_ptr + row * stride_gu + cols, mask=mask, other=0.0).to(tl.float32)
    u = tl.load(gu_ptr + row * stride_gu + n_inter + cols, mask=mask, other=0.0).to(tl.float32)
    out = g * tl.sigmoid(g) * u
    tl.store(out_ptr + row * stride_out + cols, out.to(out_ptr.dtype.element_ty), mask=mask)

## Plugging into PyTorch

In [ ]:
@torch.library.custom_op("ia::add_rmsnorm", mutates_args=())
def add_rmsnorm(x: torch.Tensor, residual: torch.Tensor, w: torch.Tensor, eps: float) -> tuple[torch.Tensor, torch.Tensor]:
    x, residual = x.contiguous(), residual.contiguous()
    n_cols = x.shape[-1]
    x2, r2 = x.view(-1, n_cols), residual.view(-1, n_cols)
    y, h = torch.empty_like(x2), torch.empty_like(x2)
    BLOCK = triton.next_power_of_2(n_cols)
    add_rmsnorm_kernel[(x2.shape[0],)](x2, r2, w, y, h, x2.stride(0), n_cols, eps,
                                       BLOCK=BLOCK, num_warps=min(max(BLOCK // 256, 1), 16))
    return y.view_as(x), h.view_as(x)

@add_rmsnorm.register_fake
def _(x, residual, w, eps):
    return torch.empty_like(x), torch.empty_like(x)

@torch.library.custom_op("ia::swiglu", mutates_args=())
def swiglu(gate_up: torch.Tensor) -> torch.Tensor:
    gate_up = gate_up.contiguous()
    n_inter = gate_up.shape[-1] // 2
    gu = gate_up.view(-1, 2 * n_inter)
    out = torch.empty((gu.shape[0], n_inter), device=gu.device, dtype=gu.dtype)
    BLOCK = 1024
    swiglu_kernel[(gu.shape[0], triton.cdiv(n_inter, BLOCK))](gu, out, n_inter, gu.stride(0), out.stride(0), BLOCK=BLOCK)
    return out.view(*gate_up.shape[:-1], n_inter)

@swiglu.register_fake
def _(gate_up):
    return gate_up.new_empty((*gate_up.shape[:-1], gate_up.shape[-1] // 2))

## Testing it properly

In [ ]:
import itertools
def ref_add_rmsnorm(x, r, w, eps):
    h = x.float() + r.float()
    y = h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + eps) * w.float()
    return y.to(x.dtype), h.to(x.dtype)

dtypes = [torch.float16] + ([torch.bfloat16] if torch.cuda.is_bf16_supported() else [])
for rows, cols, dt in itertools.product([1, 7, 128, 4096], [768, 4096, 5120, 8192], dtypes):
    x = torch.randn(rows, cols, device="cuda", dtype=dt)
    r = torch.randn_like(x); w = torch.randn(cols, device="cuda", dtype=dt)
    y, h = torch.ops.ia.add_rmsnorm(x, r, w, 1e-6)
    y_ref, h_ref = ref_add_rmsnorm(x, r, w, 1e-6)
    torch.testing.assert_close(h, h_ref); torch.testing.assert_close(y, y_ref, atol=2e-2, rtol=2e-2)
print("all shapes pass")

## Benchmarking against the alternatives

In [ ]:
def eager(x, r, w):
    h = x + r
    return torch.nn.functional.rms_norm(h, (h.shape[-1],), w, 1e-6), h
compiled = torch.compile(eager)

for rows in [4096, 8]:
    x = torch.randn(rows, 4096, device="cuda", dtype=torch.float16); r = torch.randn_like(x)
    w = torch.randn(4096, device="cuda", dtype=torch.float16)
    for name, f in [("eager", lambda: eager(x, r, w)), ("compiled", lambda: compiled(x, r, w)),
                    ("ours", lambda: torch.ops.ia.add_rmsnorm(x, r, w, 1e-6))]:
        ms = triton.testing.do_bench(f)
        print(f"rows={rows:5d} {name:9s} {ms * 1000:8.1f} us, {4 * x.numel() * 2 / ms / 1e6:6.0f} GB/s (fused traffic)")

## Where these kernels live in real engines

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-triton/fused-rmsnorm-and-swiglu-plugged-into-pytorch